In [1]:
%%time

%pip install pyspark==4.1.2 pandas numpy neo4j redis hiredis PyArrow>=15.0.0 graphframes grpcio grpcio-status zstandard

Note: you may need to restart the kernel to use updated packages.
CPU times: user 65.5 ms, sys: 11.5 ms, total: 77 ms
Wall time: 2.05 s


# FalkorDB VS Neo4J API

In [2]:
# Open the Neo4j driver and prepare the query helper functions.
import pandas as pd
from neo4j import GraphDatabase
driver_n4j=None
NEO4J_DB = "neo4j"
try:
  # Neo4j connection settings
  NEO4J_URI      = 'neo4j://debianmagiinferencecpu.local.lan:7687' 
  NEO4J_USER = "test"
  NEO4J_PASSWORD = "password"

  driver_n4j = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
  driver_n4j.verify_connectivity()
except Exception as e:
  
  print(f"NEO4J_URI: {NEO4J_URI}")
  try:
    NEO4J_URI      = 'neo4j://10.0.0.46:7687' 
    driver_n4j = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver_n4j.verify_connectivity()
  except Exception as e:
    print(f"NEO4J_URI: {NEO4J_URI}")
    print("Please check your connection settings and ensure the Neo4j server is running.")
    raise ConnectionError("Failed to connect to Neo4j. Please check your connection settings and ensure the Neo4j server is running.")

print("Connected to Neo4j")

Connected to Neo4j


In [3]:
# Open the Neo4j driver and prepare the query helper functions.
import pandas as pd
from redis import Redis
driver_fbd=None
DATABASE = "falkordb"
try:
  FALKORDB_HOST = "10.0.0.79"  # o l'IP del tuo server FalkorDB
  FALKORDB_PORT = 6379
  driver_fbd = Redis(
      host=FALKORDB_HOST,
      port=FALKORDB_PORT,
      decode_responses=True,
      socket_connect_timeout=10,
      socket_timeout=30
  )
  driver_fbd.ping()

except Exception as e:
  
  print(f"Redis: FalkorDB: {FALKORDB_HOST}:{FALKORDB_PORT}")
  print("Please check your connection settings and ensure the FalkorDB server is running.")
  raise ConnectionError("Failed to connect to FalkorDB. Please check your connection settings and ensure the FalkorDB server is running.")

print("Connected to FalkorDB")

Connected to FalkorDB


In [4]:
import pandas as pd
from IPython.display import display


def neo4j_query_df(cypher: str, params: dict | None = None) -> pd.DataFrame:
    with driver_n4j.session(database=NEO4J_DB) as session:
        return pd.DataFrame([r.data() for r in session.run(cypher, params or {})])

def falkor_query_df(cypher: str) -> pd.DataFrame:
    raw = driver_fbd.execute_command("GRAPH.RO_QUERY" , DATABASE, cypher)
    cols = [str(c[1]) if isinstance(c, (list, tuple)) and len(c) > 1 else str(c) for c in raw[0]]
    return pd.DataFrame(raw[1], columns=cols)

def compare_dfS(df1: pd.DataFrame, df2: pd.DataFrame, head: int = 4) -> bool | None:
    if list(df1.columns) == list(df2.columns):
        return df1.head(head).reset_index(drop=True).equals(df2.head(head).reset_index(drop=True))
    else:
        print("Columns do not match:")
        print(f"Neo4j columns: {list(df1.columns)}")
        print(f"FalkorDB columns: {list(df2.columns)}")


def compare_query_graphdb(neo4j_cypher: str, falkor_cypher: str | None = None, head: int = 4):
    neo_df = neo4j_query_df(neo4j_cypher)
    fbd_df = falkor_query_df(falkor_cypher)
    eq = compare_dfS(neo_df, fbd_df, head)
    print(f"Head equality (first {head} rows): {eq}")
    if not eq:
        print(f"Neo4j rows: {len(neo_df)}")
        display(neo_df.head(head))
        print(f"FalkorDB rows: {len(fbd_df)}")
        display(fbd_df.head(head))


def compare_query_tabular_pd_spark(result_df_1_pandas: pd.DataFrame, result_df_2_spark: pd.DataFrame | None = None, head: int = 4):
    eq = compare_dfS(result_df_1_pandas, result_df_2_spark, head)
    print(f"Head equality (first {head} rows): {eq}")
    if not eq:
        print(f"Pandas rows: {len(result_df_1_pandas)}")
        display(result_df_1_pandas.head(head))
        print(f"Spark rows: {len(result_df_2_spark)}")
        display(result_df_2_spark.head(head))

In [5]:
#==========================================================================================
#   COMPARE: Node label cardinality
#==========================================================================================


q: dict[str, str] = {
    "neo4j": """
        MATCH (n)
        UNWIND labels(n) AS lbl
        RETURN lbl AS node_label, count(*) AS count
        ORDER BY count DESC, node_label
    """,
    "falkor": """
        MATCH (n)
        UNWIND labels(n) AS lbl
        RETURN lbl AS node_label, count(*) AS count
        ORDER BY count DESC, node_label
    """,
}

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)


Head equality (first 4 rows): True


In [6]:
#==========================================================================================
#   COMPARE: Relationship types cardinality
#==========================================================================================


q:dict[str, str] = { 
        "neo4j": """
            MATCH ()-[r]->()
            RETURN type(r) AS rel_type, count(*) AS count
            ORDER BY count DESC, rel_type
        """,
        "falkor": """
            MATCH ()-[r]->()
            RETURN type(r) AS rel_type, count(*) AS count
            ORDER BY count DESC, rel_type
        """,
}

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)

Head equality (first 4 rows): True


In [7]:
#==========================================================================================
#   COMPARE: Route relationships count
#==========================================================================================

q:dict[str, str] =     {
        "neo4j": "MATCH (:Airport)-[r:ROUTE_TO]->(:Airport) RETURN count(r) AS route_relationships",
        "falkor": "MATCH (:Airport)-[r:ROUTE_TO]->(:Airport) RETURN count(r) AS route_relationships",
    }

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)

Head equality (first 4 rows): True


In [8]:
#==========================================================================================
#   COMPARE: Top routes by flights
#==========================================================================================


q:dict[str, str] =     {

        "neo4j": """
            MATCH (o:Airport)-[r:ROUTE_TO]->(d:Airport)
            RETURN o.iata AS origin,
                   d.iata AS dest,
                   r.flights AS flights
            ORDER BY flights DESC, origin, dest
            LIMIT 15
        """,
        "falkor": """
            MATCH (o:Airport)-[r:ROUTE_TO]->(d:Airport)
            RETURN o.iata AS origin,
                   d.iata AS dest,
                   r.flights AS flights
            ORDER BY flights DESC, origin, dest
            LIMIT 15
        """,    }

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)

Head equality (first 4 rows): True


In [9]:
#==========================================================================================
#   COMPARE: Top routes by average delay (flights >= 10)
#==========================================================================================


q:dict[str, str] =     {

        "neo4j": """
            MATCH (o:Airport)-[r:ROUTE_TO]->(d:Airport)
            WHERE r.flights >= 10
            RETURN o.iata AS origin,
                   d.iata AS dest,
                   r.flights AS flights,
                   ceil(r.avg_delay) AS avg_delay
            ORDER BY avg_delay DESC, origin, dest
            LIMIT 5
        """,
        "falkor": """
            MATCH (o:Airport)-[r:ROUTE_TO]->(d:Airport)
            WHERE r.flights >= 10
            RETURN o.iata AS origin,
                   d.iata AS dest,
                   r.flights AS flights,
                   ceil(r.avg_delay) AS avg_delay
            ORDER BY avg_delay DESC, origin, dest
            LIMIT 5
        """,

    }

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)

Head equality (first 4 rows): False
Neo4j rows: 5


,origin,dest,flights,avg_delay
0,ATL,BWI,12,73.0
1,SFO,ORD,10,20.0
2,LAS,LAX,12,19.0
3,LAX,PHX,13,18.0


FalkorDB rows: 5


,origin,dest,flights,avg_delay
0,ATL,BWI,12,73
1,SFO,ORD,10,20
2,LAS,LAX,12,19
3,LAX,PHX,13,18


In [10]:
#==========================================================================================
#   COMPARE: Top in-degree airports
#==========================================================================================


q: dict[str, str] = {
    "neo4j": """
        MATCH (a:Airport)
        OPTIONAL MATCH ()-[inn:ROUTE_TO]->(a)
        WITH a, count(inn) AS in_degree
        RETURN a.iata AS iata, in_degree
        ORDER BY in_degree DESC, iata
        LIMIT 10
    """,
    "falkor": """
        MATCH (a:Airport)
        OPTIONAL MATCH ()-[inn:ROUTE_TO]->(a)
        WITH a, count(inn) AS in_degree
        RETURN a.iata AS iata, in_degree
        ORDER BY in_degree DESC, iata
        LIMIT 10
    """,
}

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)

Head equality (first 4 rows): True


In [11]:
#==========================================================================================
#   COMPARE: out-degree airports
#==========================================================================================


q: dict[str, str] = {
        "neo4j": """
            MATCH (a:Airport)
            OPTIONAL MATCH (a)-[out:ROUTE_TO]->()
            WITH a, count(out) AS out_degree
            RETURN a.iata AS iata, out_degree
            ORDER BY out_degree DESC, iata
            LIMIT 10
        """,
        "falkor": """
            MATCH (a:Airport)
            OPTIONAL MATCH (a)-[out:ROUTE_TO]->()
            WITH a, count(out) AS out_degree
            RETURN a.iata AS iata, out_degree
            ORDER BY out_degree DESC, iata
            LIMIT 10
        """,
}

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)

Head equality (first 4 rows): True


## Algorithm

In [12]:
#==========================================================================================
#   COMPARE: community analisys
#==========================================================================================


q: dict[str, str] = {
        "neo4j": """
            CALL gds.louvain.stream('airport_route_native', {relationshipWeightProperty: 'flights'})
            YIELD nodeId, communityId
            RETURN gds.util.asNode(nodeId).iata AS iata, communityId
            ORDER BY communityId, iata
            LIMIT 20
        """,
        "falkor": """
            CALL algo.labelPropagation({
                nodeLabels: ['Airport'],
                relationshipTypes: ['ROUTE_TO']
            })
            YIELD node, communityId
            RETURN node.iata AS iata, communityId
            ORDER BY  communityId, iata
            LIMIT 20
        """,
}

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
    head=10
)

Head equality (first 10 rows): False
Neo4j rows: 20


,iata,communityId
0,PSG,237
1,WRG,237
2,ABE,266
3,ABR,266
4,ALO,266
5,ATW,266
6,AVP,266
7,AZO,266
8,BGM,266
9,BJI,266


FalkorDB rows: 20


,iata,communityId
0,YAK,1
1,HIB,2
2,INL,2
3,MEI,2
4,WRG,3
5,ABE,14
6,ABI,14
7,ABQ,14
8,ABR,14
9,ACK,14


In [13]:
#==========================================================================================
#   COMPARE: centrality analisys
#==========================================================================================


q: dict[str, str] = {
        "neo4j": """
            CALL gds.betweenness.stream('airport_route_native', {
            relationshipWeightProperty: 'flights'})
            YIELD nodeId, score
            RETURN gds.util.asNode(nodeId).iata AS iata, ceil(score) AS betweenness
            ORDER BY betweenness DESC
            LIMIT 20
        """,
        "falkor": """
            CALL algo.betweenness({
                nodeLabels: ['Airport'],
                relationshipTypes: ['ROUTE_TO'],
                samplingSeed: 667
            })
            YIELD node, score
            RETURN node.iata AS iata, ceil(score) AS betweenness
            ORDER BY betweenness DESC
            LIMIT 20
        """,
}

compare_query_graphdb(
    neo4j_cypher=q["neo4j"],
    falkor_cypher=q["falkor"],
)

Head equality (first 4 rows): False
Neo4j rows: 20


,iata,betweenness
0,DFW,11205.0
1,ATL,10525.0
2,ORD,10285.0
3,DEN,6752.0


FalkorDB rows: 20


,iata,betweenness
0,ATL,1144
1,ORD,992
2,DFW,898
3,DEN,775


# Pandas VS Spark

In [14]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import os
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import SparkContext

WORK_TMP_DIR = str(Path('/tmp/.spark-tmp').resolve())
os.makedirs(WORK_TMP_DIR, exist_ok=True)

def create_spark_session(
    app_name: str = "DiffAPIspark",
    master_url: str = "spark://10.0.0.79:7077", # local cluster URL, for local execution use "local[*]"
    driver_bind_address: str = "0.0.0.0",
    logging_level: str = "ERROR",
) -> "SparkSession":

    #SparkSession.builder.getOrCreate().stop() # Stop any existing Spark session to avoid conflicts
    try:
        spark = SparkSession.builder.getOrCreate()
        spark.stop()
        spark._sc._gateway.shutdown()
        spark._sc._gateway.proc.stdin.close()
        SparkContext._gateway = None
        SparkContext._jvm = None
    except Exception:
        pass

    spark = (
        SparkSession.builder
        .appName(app_name)
        .master(master_url)
        .config("spark.driver.bindAddress", driver_bind_address)
        .config("spark.local.dir", WORK_TMP_DIR)
        .config("spark.driver.extraJavaOptions", f"-Djava.io.tmpdir={WORK_TMP_DIR}")
        .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.9.3") # graphframes
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel(logging_level)
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    spark.conf.set("spark.sql.execution.arrow.pyspark.fallback.enabled", "false")
    return spark




In [15]:
def format_hhmm_pd(value):
    if pd.isna(value):
        return pd.NA
    try:
        value = int(value)
    except Exception:
        return pd.NA
    if value == 2400:
        value = 0
    if value < 0 or value > 2359:
        return pd.NA
    s = f"{value:04d}"
    hh, mm = int(s[:2]), int(s[2:])
    if hh > 23 or mm > 59:
        return pd.NA
    return pd.Timestamp(2000, 1, 1, hh, mm).time()


def format_hhmm_pd_spark(col_name: str):
    """
    Converts HHMM integer format to a native Spark Timestamp.
    Returns None for invalid or null values.
    """
    raw = F.col(col_name).cast("int")
    normalized = F.when(raw == 2400, F.lit(0)).otherwise(raw)
    hh = F.floor(normalized / 100)
    mm = normalized % 100
    valid = (
        normalized.isNotNull()
        & (normalized >= 0)
        & (normalized <= 2359)
        & (hh <= 23)
        & (mm <= 59)
    )
    as_ts = F.to_timestamp(
        F.concat(F.lit("2000-01-01 "), F.lpad(normalized.cast("string"), 4, "0")),
        "yyyy-MM-dd HHmm",
    )
    return F.when(valid, F.date_format(as_ts, "HH:mm:ss")).otherwise(F.lit(None))
    #return F.when(valid, as_ts).otherwise(F.lit(None))

In [16]:
test_cases = [1430, 0, 2400, 5, "830", 2401, 1260, -5, pd.NA, None, np.nan]

df_test = pd.DataFrame({"value": test_cases})
df_pd = pd.DataFrame(df_test["value"].apply(format_hhmm_pd))

import datetime

df_pd["value"] = df_pd["value"].apply(
    lambda x: x.strftime("%H:%M:%S") if pd.notna(x) and isinstance(x, datetime.time) else x
)

spark = create_spark_session()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")


df_test["value"] = df_test["value"].map(
    lambda v: None if pd.isna(v) else str(v)
)

df_spark = spark.createDataFrame(df_test)
df_spark = df_spark.withColumn("value", format_hhmm_pd_spark("value"))
df_spark = df_spark.toPandas()

compare_query_tabular_pd_spark(
    result_df_1_pandas=df_pd,
    result_df_2_spark=df_spark,
    head=4
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/10 16:28:53 WARN Utils: Your hostname, pvenas10, resolves to a loopback address: 127.0.1.1; using 10.0.0.100 instead (on interface vmbr0)
26/09/10 16:28:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 16:28:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/10 16:29:03 WARN Utils: Your hostname, pvenas10, resolves to a loopback address: 127.0.1.1; using 10.0.0.100 instead (on interface vmbr0)
26/09/10 16:29:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings 

Head equality (first 4 rows): True


# Spark VS Sail

In [17]:
%%time


def create_sail_session(
    app_name: str = "DiffAPIsail",
    master_url: str = "sc://10.0.0.79:6066",
    driver_bind_address: str = "0.0.0.0",
) -> "SparkSession":
    try:
        spark = SparkSession.builder.getOrCreate()
        spark.stop()
        spark._sc._gateway.shutdown()
        spark._sc._gateway.proc.stdin.close()
        SparkContext._gateway = None
        SparkContext._jvm = None
    except Exception:
        pass
    return (
        SparkSession.builder
        .appName(app_name)
        .remote(master_url)
        .config("spark.driver.bindAddress", driver_bind_address)
        .getOrCreate()
    )

CPU times: user 17 μs, sys: 0 ns, total: 17 μs
Wall time: 24.1 μs


In [19]:
import time



# ============================================================
# CONFIGURAZIONE
# ============================================================
N = 5_000_000          # righe base -> alza a 10_000_000 se troppo veloce
PARTITIONS = 8         # parallelismo
SEED = 42

def run_benchmark(session, ) -> float:
    """
    Esegue il benchmark sulla sessione passata (spark o sail)
    e restituisce il tempo in secondi.
    """
    from pyspark.sql import functions as F
    from pyspark.sql.window import Window

    # generate synthetic DataFrame
    df = (
        session.range(0, N, numPartitions=PARTITIONS)
        .withColumn("id", F.col("id"))
        .withColumn("grp", (F.col("id") % 100).cast("int"))
        .withColumn("val", (F.rand(seed=SEED) * 1000).cast("double"))
        .withColumn("cat", (F.col("id") % 5).cast("int"))
    )

    # aggregate
    agg = (
        df.groupBy("grp", "cat")
          .agg(
              F.sum("val").alias("sum_val"),
              F.avg("val").alias("avg_val"),
              F.stddev("val").alias("std_val"),
              F.count("*").alias("cnt"),
              F.approx_count_distinct("val").alias("approx_dc"),
          )
    )


    # Self-join product
    joined = (
        agg.alias("a")
        .join(agg.alias("b"), on="grp", how="inner")
        .select(
            F.col("grp"),                # <-- senza qualificatore
            F.col("a.cat").alias("cat_a"),
            F.col("b.cat").alias("cat_b"),
            (F.col("a.sum_val") - F.col("b.sum_val")).alias("delta"),
        )
    )

    # Window function + orderBy
    w = Window.partitionBy("grp").orderBy(F.col("delta").desc())
    ranked = (
        joined
        .withColumn("rk", F.row_number().over(w))
        .filter(F.col("rk") <= 50)
    )


    # get benchmark start time
    t0 = time.perf_counter()

    # cache + count 
    ranked.cache()
    n_rows = ranked.count()

    # second action: aggregate sum of delta
    total = ranked.agg(F.sum("delta")).collect()[0][0]

    # end benchmark timer
    t1 = time.perf_counter()

    elapsed = t1 - t0

    # cleanup
    ranked.unpersist()
    return elapsed


# ============================================================
# ESECUZIONE
# ============================================================
spark = create_spark_session("SPARK")
t_spark = run_benchmark(spark, )

sail = create_sail_session("SAIL")
t_sail  = run_benchmark(sail,  )

print("\n===== RISULTATO BENCHMARK =====")
print(f"Spark : {t_spark:.3f} s")
print(f"Sail  : {t_sail:.3f} s")
print(f"Speedup Sail/Spark: {t_spark / t_sail:.2f}x")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/10 16:29:49 WARN Utils: Your hostname, pvenas10, resolves to a loopback address: 127.0.1.1; using 10.0.0.100 instead (on interface vmbr0)
26/09/10 16:29:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/media/simone/USB_projects/Usefull/ATTIVI/Esericizi/Big_data/venv_310/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/simone/.ivy2.5.2/cache
The jars for the packages stored in: /home/simone/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8c762597-1d8f-4c2c-baf4-195dec81fa8e;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.9.3 in central
:: resolution report :: resolve 224ms :: artifacts dl 6ms
	:: modules in use:
	io.graphframes#graphframes-sp


===== RISULTATO BENCHMARK =====
Spark : 25.537 s
Sail  : 0.586 s
Speedup Sail/Spark: 43.59x
